In [ ]:
from sklearn.datasets import load_breast_cancer
import numpy as np
import matplotlib.pyplot as plt

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import cross_val_score

url = 'https://github.com/JLZml/Credit-Scoring-Data-Sets/raw/refs/heads/master/3.%20Kaggle/Give%20Me%20Some%20Credit/cs-training.csv'
df = pd.read_csv(url)

df.drop('Unnamed: 0', axis=1, inplace=True)

print(df.describe())
print(df.head())
print(df.info())

print(df.shape)
print(df.isna().sum())

df['SeriousDlqin2yrs'].value_counts(dropna = False)
df['NumberOfTimes90DaysLate'].value_counts(dropna = False)

print(df['age'].shape[0])

print(df[df['NumberOfTimes90DaysLate'] != 98].shape[0])
print(df[df['RevolvingUtilizationOfUnsecuredLines'] > 2].shape[0])

df = df[(df['age'] != 0) & (df['NumberOfTimes90DaysLate'] != 98) & (df['RevolvingUtilizationOfUnsecuredLines'] <= 2)]

median_numDep = df['NumberOfDependents'].median()
df['NumberOfDependents'] = df['NumberOfDependents'].fillna(median_numDep)

mean_monInc = df['MonthlyIncome'].mean()
df['MonthlyIncome'] = df['MonthlyIncome'].fillna(mean_monInc)

print(df.isna().sum())
print(df.shape)

TotalPastDue = df['NumberOfTime30-59DaysPastDueNotWorse'] + df["NumberOfTime60-89DaysPastDueNotWorse"] + df["NumberOfTimes90DaysLate"]

df['TotalPastDue'] = TotalPastDue

df['IncomePerDependent'] = df['MonthlyIncome'] / (df['NumberOfDependents'] + 1)

X = df.drop('SeriousDlqin2yrs' , axis = 1)
y = df['SeriousDlqin2yrs']

scaler = StandardScaler()

X_train, X_test, y_train, y_test = train_test_split(X, y,stratify=y)

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression()

scores = cross_val_score(model, X_train_scaled, y_train, cv=5)
print(scores)
print(scores.mean())

scores_f1 = cross_val_score(model, X_train_scaled, y_train, cv =5,scoring = 'f1')
print(scores_f1)
print(scores_f1.mean())

scores_recall = cross_val_score(model, X_train_scaled, y_train, cv =5,scoring = 'recall')
print(scores_recall)
print(scores_recall.mean())

model_balanced = LogisticRegression(class_weight='balanced')

scores_recall_balanced = cross_val_score(model_balanced, X_train_scaled, y_train, cv=5, scoring='recall')
print(scores_recall_balanced.mean())

scores_f1_balanced = cross_val_score(model_balanced, X_train_scaled, y_train, cv=5, scoring='f1')
print(scores_f1_balanced.mean())

model_class = LogisticRegression(class_weight='balanced')
model_class.fit(X_train_scaled, y_train)

pred = model_class.predict(X_test_scaled)
print(pred)

print("Финальные метрики на test:")
print("Accuracy:", accuracy_score(y_test, pred))
print("Precision:", precision_score(y_test, pred))
print("Recall:", recall_score(y_test, pred))
print("F1:", f1_score(y_test, pred))

       SeriousDlqin2yrs  RevolvingUtilizationOfUnsecuredLines            age  \
count     150000.000000                         150000.000000  150000.000000   
mean           0.066840                              6.048438      52.295207   
std            0.249746                            249.755371      14.771866   
min            0.000000                              0.000000       0.000000   
25%            0.000000                              0.029867      41.000000   
50%            0.000000                              0.154181      52.000000   
75%            0.000000                              0.559046      63.000000   
max            1.000000                          50708.000000     109.000000   

       NumberOfTime30-59DaysPastDueNotWorse      DebtRatio  MonthlyIncome  \
count                         150000.000000  150000.000000   1.202690e+05   
mean                               0.421033     353.005076   6.670221e+03   
std                                4.192781    2